# Early (ectopic) beats from RR intervals — feasibility on MIT-BIH Arrhythmia (Notebook 10)
**Bektur Asilov · COMSE-23 · Ala-Too**

Frequent premature beats (PVC — ventricular, PAC — atrial) are common, often harmless, but a high burden is worth
showing to a cardiologist. Question: can the device count them from the **RR intervals it already measures**
(same R-peak detector as the firmware), without a new model?

Rule fixed **before** the run (records with pacemakers 102, 104, 107, 217 are excluded, as in ANSI/AAMI EC57):
* RRᵢ = time from beat i−1 to beat i; reference = median of the 8 previous RR;
* beat i is **early** if RRᵢ < 0.80 × reference **and** the next interval RRᵢ₊₁ > 1.15 × RRᵢ (rhythm slows back);
* not judged while the 8 reference intervals are themselves irregular (mean |ΔRR| / mean RR > 0.10 — e.g. AF);
* truth: cardiologist labels, ventricular = V, E; supraventricular = A, a, J, S (AAMI classes);
  detected beats matched to labelled beats within 150 ms;
* reported: sensitivity for V and for S separately, PPV (flagged beats that are V or S), and per-record burden
  (% of beats) detected vs labelled, including agreement on "burden ≥ 10 %".
* RR alone cannot tell ventricular from atrial; that is reported, not hidden.

> Kaggle: Internet **On**, CPU.

In [ ]:
import os, time, json, subprocess, sys
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
DB = os.environ.get('MITDB_DIR')
RECS = [100,101,102,103,104,105,106,107,108,109,111,112,113,114,115,116,117,118,119,121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,219,220,221,222,223,228,230,231,232,233,234]
if not DB:
    DB = '/kaggle/tmp/mitdb'; os.makedirs(DB, exist_ok=True)
    for r in RECS:
        for ext in ('hea', 'dat', 'atr'):
            for base in ('https://physionet-open.s3.amazonaws.com/mitdb/1.0.0/', 'https://physionet.org/files/mitdb/1.0.0/'):
                if subprocess.run(['curl', '-fsSL', '--max-time', '300', '-o', f'{DB}/{r}.{ext}', base + f'{r}.{ext}']).returncode == 0: break
            else:
                raise RuntimeError(f'download failed: {r}.{ext}')
print('records:', len(RECS))

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

BEAT = set('NLRBAaJSVrFejnE/fQ?')

In [ ]:
PACED = {102, 104, 107, 217}
V, S = set('VE'), set('AaJS')
def early_flags(t):
    rr = np.diff(t) / FS                       # rr[k] = interval ending at beat k+1
    flags = np.zeros(len(t), bool)
    for k in range(8, len(rr) - 1):
        refw = rr[k - 8:k]; ref = np.median(refw)
        if np.mean(np.abs(np.diff(refw))) / np.mean(refw) > 0.10: continue
        if rr[k] < 0.80 * ref and rr[k + 1] > 1.15 * rr[k]: flags[k + 1] = True
    return flags
rows = []
for r in [x for x in RECS if x not in PACED and os.path.exists(f'{DB}/{x}.dat')]:
    rec = wfdb.rdrecord(f'{DB}/{r}', channels=[0]); ann = wfdb.rdann(f'{DB}/{r}', 'atr')
    x = to_125(rec.p_signal[:, 0], rec.fs)
    lab = [(s * FS / rec.fs, sym) for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT]
    ref_t = np.array([a for a, _ in lab]); ref_c = np.array(['V' if s in V else 'S' if s in S else 'N' for _, s in lab])
    det = detect_r(x).astype(float); fl = early_flags(det)
    j = np.clip(np.searchsorted(ref_t, det), 1, len(ref_t) - 1)
    near = np.where(np.abs(ref_t[j - 1] - det) < np.abs(ref_t[j] - det), j - 1, j)
    ok = np.abs(ref_t[near] - det) <= 0.15 * FS
    cls = np.where(ok, ref_c[near], 'X')         # class of the labelled beat each detected beat matches (X = none)
    rows.append(dict(record=r, beats=len(ref_t), ref_V=int((ref_c == 'V').sum()), ref_S=int((ref_c == 'S').sum()),
                     det_beats=len(det), flagged=int(fl.sum()), flagged_V=int((fl & (cls == 'V')).sum()),
                     flagged_S=int((fl & (cls == 'S')).sum()), flagged_N=int((fl & (cls == 'N')).sum()),
                     flagged_X=int((fl & (cls == 'X')).sum())))
R = pd.DataFrame(rows); R.to_csv(f'{OUT_DIR}/ectopy_per_record.csv', index=False)
R['burden_ref'] = (R.ref_V + R.ref_S) / R.beats; R['burden_det'] = R.flagged / R.det_beats
from scipy.stats import spearmanr
hi_ref, hi_det = R.burden_ref >= 0.10, R.burden_det >= 0.10
Sm = {'records': len(R), 'beats': int(R.beats.sum()), 'ref_V': int(R.ref_V.sum()), 'ref_S': int(R.ref_S.sum()),
      'sens_V': R.flagged_V.sum() / R.ref_V.sum(), 'sens_S': R.flagged_S.sum() / R.ref_S.sum(),
      'ppv': (R.flagged_V.sum() + R.flagged_S.sum()) / R.flagged.sum(), 'flagged': int(R.flagged.sum()),
      'burden_spearman': spearmanr(R.burden_ref, R.burden_det).correlation,
      'burden_mae_pct_points': float((R.burden_ref - R.burden_det).abs().mean() * 100),
      'records_burden_ge10_ref': int(hi_ref.sum()), 'of_them_detected_ge10': int((hi_ref & hi_det).sum()),
      'false_ge10': int((~hi_ref & hi_det).sum())}
json.dump(Sm, open(f'{OUT_DIR}/ectopy_summary.json', 'w'), indent=1, default=float)
print(json.dumps(Sm, indent=1, default=lambda v: round(float(v), 4)))
pd.set_option('display.width', 200)
print(R.assign(burden_ref=(R.burden_ref * 100).round(1), burden_det=(R.burden_det * 100).round(1)).to_string(index=False))